In [1]:
import pandas as pd
import numpy as np

import yfinance as yf

In [2]:
with open("../data/tickers.txt", "r") as f:
    tickers = [line.strip() for line in f]

In [3]:
start = "2021-12-31"
end = "2023-01-01"

prices = yf.download(
    tickers,
    start,
    end,
    auto_adjust=False,
    progress=False,
)


2 Failed downloads:
['CHK', 'BGNE']: YFTzMissingError('possibly delisted; no timezone found')


In [4]:
prices.shape

(252, 810)

In [5]:
adj_close = prices["Adj Close"]
adj_close = adj_close.dropna(axis=1)
adj_close.shape

(252, 131)

In [6]:
hundredstocks = np.random.choice(adj_close.columns, size=100, replace=False)
P = adj_close[hundredstocks]

In [7]:
R = P.pct_change().dropna()
R.shape

(251, 100)

In [8]:
S = np.cov(R, rowvar=False)
S.shape

(100, 100)

Here, we compute the sample covariance matrix S which contains the covariances of 100 randomly selected tickers.

In [9]:
ones = np.ones((100, 1))

Sinv = np.linalg.inv(S)

numerator = Sinv @ ones
denominator = ones.T @ Sinv @ ones

w = numerator / denominator
w.sum()

0.9999999999999903

Using the minimum variance portfolio solution, we compute the weights of each ticker and store it in a column vector, where each element represents the optimal allocation to a stock.

In [10]:
weights = pd.Series(w.flatten(), index=R.columns)

top5 = weights.sort_values(ascending=False).head(5)
bottom5 = weights.sort_values(ascending=True).head(5)

top5, bottom5

(Ticker
 GOOG     0.446545
 FWONA    0.229602
 KDP      0.172443
 SNY      0.137534
 VRSK     0.108883
 dtype: float64,
 Ticker
 GOOGL   -0.340590
 MCHP    -0.168871
 FWONK   -0.164704
 INTU    -0.136082
 TSCO    -0.108586
 dtype: float64)

The column vector of portfolio weights is flattened and converted into a Pandas Series so that each weight is labeled with its corresponding ticker for clear interpretation. The weights are then sorted to identify the top five and bottom five stocks based on their portfolio allocations.

In [11]:
I = np.eye(100)
variance = np.mean(np.diag(S))
variance

0.0010698625166902422

We extract the diagonal elements of the covariance matrix S, which represent the individual variances of each stock’s daily returns. We then compute the mean of these diagonal values to obtain the average variance across all stocks in the portfolio.

In [12]:
alphas = np.linspace(0, 1, 11)

S_alpha_dict = {}

for a in alphas:
    S_alpha = a * S + (1 - a) * variance * I
    S_alpha_dict[a] = S_alpha

Here, I store each of the 11 matrices S(α) in a Python dictionary where the key is α and the value is the matrix S(α). This makes it easy to retrieve any matrix later using S_alpha_dict[alpha].

In [13]:
weights_alpha_dict = {}

for a in alphas:
    S_alpha = S_alpha_dict[a]
    
    Sinv = np.linalg.inv(S_alpha)
    
    numerator = Sinv @ ones
    denominator = ones.T @ Sinv @ ones
    w_alpha = numerator / denominator
    
    weights_alpha_dict[a] = w_alpha

In [14]:
w_alpha.shape

(100, 1)

In [15]:
weights_1 = pd.Series(weights_alpha_dict[1.0].flatten(), index=R.columns)
top5 = weights_1.sort_values(ascending=False).head(5)
bottom5 = weights_1.sort_values(ascending=True).head(5)

top5, bottom5

(Ticker
 GOOG     0.446545
 FWONA    0.229602
 KDP      0.172443
 SNY      0.137534
 VRSK     0.108883
 dtype: float64,
 Ticker
 GOOGL   -0.340590
 MCHP    -0.168871
 FWONK   -0.164704
 INTU    -0.136082
 TSCO    -0.108586
 dtype: float64)

Using the sample covariance matrix S, where α=1, we compute the minimum variance portfolio weights. These weights represent the allocation to each stock that minimizes total portfolio variance. We then sort the weights to identify the top 5 stocks with the largest positive weights and the bottom 5 stocks with the most negative weights.

Positive weights indicate stocks that help reduce portfolio risk, while negative weights represent short positions used to hedge risk arising from correlations between assets.

With a weight of -0.5807, Google is assigned a large short position by the minimum-variance portfolio. This implies that because of its volatility or correlation with other assets, Google significantly increases the risk of the entire portfolio, according to the sample covariance matrix. When using the unregularized estimator, the sensitivity of the minimum-variance solution is highlighted by the weight's large magnitude, which reflects instability in the inverse covariance matrix.

In [16]:
weights_series_dict = {}

for a in alphas:
    weights_series_dict[a] = pd.Series(
        weights_alpha_dict[a].flatten(),
        index=R.columns
    )

In [17]:
summary_rows = []

for a in alphas:
    w = weights_series_dict[a]
    
    top5 = (w.sort_values(ascending=False).head(5) * 100).round(2)
    bottom5 = (w.sort_values(ascending=True).head(5) * 100).round(2)
    
    summary_rows.append({
        "α": a,
        "top5": list(top5.index),
        "top5 weights": list(top5.values),
        "bottom5": list(bottom5.index),
        "bottom5 weights": list(bottom5.values)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,α,top5,top5 weights,bottom5,bottom5 weights
0,0.0,"[WMG, ADBE, GFS, FWONK, LNT]","[1.0, 1.0, 1.0, 1.0, 1.0]","[WMG, FWONK, LNT, ENTG, CCEP]","[1.0, 1.0, 1.0, 1.0, 1.0]"
1,0.1,"[UTHR, SNY, KDP, LNT, GILD]","[2.52, 2.51, 2.51, 2.45, 2.35]","[MELI, TTD, MDB, APP, DKNG]","[-1.12, -1.02, -0.9, -0.86, -0.83]"
2,0.2,"[KDP, SNY, UTHR, LNT, GILD]","[3.23, 3.16, 3.14, 2.94, 2.9]","[MELI, TTD, APP, MDB, DKNG]","[-2.18, -1.49, -1.47, -1.3, -1.27]"
3,0.3,"[KDP, SNY, UTHR, GILD, NBIX]","[3.8, 3.62, 3.53, 3.28, 3.23]","[MELI, APP, AMD, TTD, CELH]","[-2.98, -1.8, -1.7, -1.56, -1.52]"
4,0.4,"[KDP, SNY, UTHR, GILD, NBIX]","[4.36, 4.03, 3.81, 3.61, 3.61]","[MELI, AMD, APP, MPWR, CELH]","[-3.7, -2.17, -2.05, -1.73, -1.67]"
5,0.5,"[KDP, SNY, UTHR, NBIX, GILD]","[4.97, 4.44, 4.05, 3.96, 3.94]","[MELI, AMD, APP, ASML, MPWR]","[-4.37, -2.61, -2.29, -2.05, -1.97]"
6,0.6,"[KDP, SNY, NBIX, GILD, UTHR]","[5.73, 4.92, 4.32, 4.31, 4.27]","[MELI, AMD, ASML, APP, INTU]","[-5.04, -3.04, -2.59, -2.54, -2.39]"
7,0.7,"[KDP, SNY, GILD, NBIX, ERIE]","[6.75, 5.52, 4.75, 4.69, 4.56]","[MELI, AMD, INTU, ASML, APP]","[-5.73, -3.54, -3.33, -3.24, -2.83]"
8,0.8,"[KDP, SNY, GILD, NTES, NBIX]","[8.29, 6.39, 5.34, 5.15, 5.1]","[MELI, INTU, AMD, ASML, FTNT]","[-6.5, -4.69, -4.16, -4.06, -3.55]"
9,0.9,"[KDP, SNY, GILD, HON, NTES]","[11.01, 7.94, 6.33, 6.31, 6.0]","[MELI, INTU, FTNT, ASML, AMD]","[-7.45, -7.02, -5.62, -5.26, -5.09]"


When α = 0, the covariance matrix reduces to $S_0 = \hat{\sigma}^2 I$. This assumes all assets have identical variance and zero correlation. In this case, the minimum-variance portfolio assigns equal weights to all stocks. Therefore, the portfolio becomes equally weighted (1/N), and there are no extreme long or short positions.

At α = 1, the sample covariance matrix 'S' is used. This leaves our minimum variance portfolio weights very sensitive to volatility and correlation patterns in the data. Google is assigned a short position of 58.07% of our portfolio, which indicates that this stock contributes significantly to overall portfolio risk. Such extreme allocations from the weights highlight noise in our variance estimations, that may have come from estimation errors when computing the matrix.

As α tends to 1, the portfolio gradually moves from equally weighted allocations to one that fully reflects the sample covariance matrix, which increases the reliance on estimated correlations and causes the portfolio to potentially become unstable. 

An important way to compare the portfolio is through an out of sample evaluation. We assess the performance of 2023 data using weights from 2022. This can be helpful to assess whether the estimated covariance matrix generalises to new data. If portfolios constructed with certain values of α perform better in 2023, this suggests that those shrinkage levels produce more stable and reliable covariance estimates.

However, it's important to understand that minimum variance portfolios are not built to maximize expected returns, they aim to minimize total portfolio variance, which measures the overall risk arising from the stock's volatility and correlation. Therefore, evaluating the performance based solely on the average return is insufficient. A portfolio may generate high returns but also exhibit extreme volatility. Since the objective of the model is to minimise risk, such assets may recieve lower weights, even if they have strong returns.

Using daily returns only to calculate our covariance matrix gives little insight on the assets potential long-run performance. To properly assess performance, it's important to consider a several other metrics:
- Annualized average return, to measure profitability.
- Annualized volatility, to measure risk

Together, these measures allow us to assess not only how much a portfolio earns, but whether the return is achieved efficiently relative to the level of risk taken.

In [18]:
start_23 = "2022-12-31"
end_23 = "2024-01-01"

prices_23 = yf.download(
    tickers,
    start_23,
    end_23,
    auto_adjust=False,
    progress=False,
)


2 Failed downloads:
['CHK', 'BGNE']: YFTzMissingError('possibly delisted; no timezone found')


In [19]:
adj_close_23 = prices_23["Adj Close"]
adj_close_23 = adj_close_23.dropna(axis=1)
adj_close_23.shape

(250, 133)

In [20]:
A = adj_close_23[hundredstocks]

In [21]:
B = A.pct_change().dropna()
B.shape

(249, 100)

In [22]:
portfolio_returns_dict = {}

for a in weights_alpha_dict:
    w = weights_alpha_dict[a]
    portfolio_returns = B.values @ w
    portfolio_returns_dict[a] = portfolio_returns.flatten()

In [23]:
performance = []

for a in portfolio_returns_dict:
    r = portfolio_returns_dict[a]
    
    mean_return = r.mean() * 252
    volatility = r.std() * np.sqrt(252)
    sharpe = mean_return / volatility
    
    performance.append([a, mean_return, volatility, sharpe])

In [24]:
performance_df = pd.DataFrame(
    performance,
    columns=["alpha", "Annual Return", "Annual Volatility", "Sharpe Ratio"]
)

performance_df

,alpha,Annual Return,Annual Volatility,Sharpe Ratio
0,0.0,0.379594,0.169946,2.233617
1,0.1,0.184688,0.121185,1.524022
2,0.2,0.118768,0.114678,1.035669
3,0.3,0.084864,0.114046,0.744119
4,0.4,0.064388,0.114781,0.560965
5,0.5,0.051073,0.115960,0.440440
6,0.6,0.042229,0.117439,0.359586
7,0.7,0.036601,0.119416,0.306496
8,0.8,0.033713,0.122559,0.275075
9,0.9,0.034074,0.129172,0.263788


The out of sample evaluation provides clear evidence of how shrinkage can impact the portfolio's performance.

As α increases, portfolio performance deteriorates. We see how larger values of α exhibit lower Sharpe ratios and weaker performance overall. Although volatility slightly decreases at intermediate shrinkage levels, annual returns fall sharply, leading to lower Sharpe ratios. The fully unregularized portfolio does not perform well against the simpler shrinkage models.

These findings  indicate that the sample covariance used in 2022 likely contains estimation noise, and distorted the portfolio's performance outside the sample year.

In contrast, shrinkage toward the identiy matrix improves stability of the weights to new data, and produced a better risk adjusted performance.